# in memory cache 

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

set_llm_cache(InMemoryCache())
llm = ChatOpenAI(model="gpt-4o-mini")

message = [HumanMessage(content="서울 광장시장에서 가장 맛있는 길거리 음식은?")]

In [2]:
%%time
# 첫 번째 호출 - API 실제 호출
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

서울 광장시장에서 가장 인기 있는 길거리 음식으로는 다음과 같은 것들이 있습니다:

1. **떡볶이**: 매콤한 소스에 쌀떡, 어묵, 야채가 들어간 인기 있는 간식입니다.
2. **순대**: 돼지 창자에 찹쌀과 여러 재료를 채워 만든 음식으로, 간혹 미느끼한 소스와 함께 제공됩니다.
3. **호떡**: 달콤한 반죽 속에 설탕, 견과류가 들어가며, 바삭하게 구워져 겨울철에 특히 인기입니다.
4. **튀김**: 다양한 재료를 튀겨낸 간식으로, 새우튀김이나 오징어튀김이 많이 팔립니다.
5. **김밥**: 밥과 다양한 재료를 김으로 감싸서 만든 음식으로, 간편하게 먹을 수 있어 인기가 많습니다.

이외에도 다양한 길거리 음식이 있으니 광장시장에서 직접 맛보면서 다양한 음식을 즐겨보세요!
CPU times: total: 0 ns
Wall time: 5.77 s


In [3]:
%%time
# 두 번째 호출 - 캐시에서 즉시 반환
response = llm.invoke(message)
print(response.content)
# Wall time: 약 1ms (거의 0)

서울 광장시장에서 가장 인기 있는 길거리 음식으로는 다음과 같은 것들이 있습니다:

1. **떡볶이**: 매콤한 소스에 쌀떡, 어묵, 야채가 들어간 인기 있는 간식입니다.
2. **순대**: 돼지 창자에 찹쌀과 여러 재료를 채워 만든 음식으로, 간혹 미느끼한 소스와 함께 제공됩니다.
3. **호떡**: 달콤한 반죽 속에 설탕, 견과류가 들어가며, 바삭하게 구워져 겨울철에 특히 인기입니다.
4. **튀김**: 다양한 재료를 튀겨낸 간식으로, 새우튀김이나 오징어튀김이 많이 팔립니다.
5. **김밥**: 밥과 다양한 재료를 김으로 감싸서 만든 음식으로, 간편하게 먹을 수 있어 인기가 많습니다.

이외에도 다양한 길거리 음식이 있으니 광장시장에서 직접 맛보면서 다양한 음식을 즐겨보세요!
CPU times: total: 0 ns
Wall time: 1 ms


In [4]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

# 프로젝트 폴더에 .langchain.db 파일로 저장
set_llm_cache(SQLiteCache(database_path=".langchain.db"))

llm = ChatOpenAI(model="gpt-4o-mini")
message = [HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")]

In [5]:
%%time
# 첫 번째 호출 - API 실제 호출 후 파일에 저장
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

코스피 지수는 한국거래소에 상장된 주식의 시가총액을 기준으로 하여 한국 주식시장의 전반적인 동향을 나타내는 지표입니다.
CPU times: total: 0 ns
Wall time: 1.24 s


In [6]:
%%time
# 두 번째 호출 - 파일에서 즉시 반환 (프로그램 재시작 후에도 동일)
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~5ms

코스피 지수는 한국거래소에 상장된 주식의 시가총액을 기준으로 하여 한국 주식시장의 전반적인 동향을 나타내는 지표입니다.
CPU times: total: 0 ns
Wall time: 2 ms


# redis 

In [12]:
import os
from dotenv import load_dotenv
from langchain_community.cache import RedisSemanticCache
from langchain_openai import OpenAIEmbeddings

load_dotenv(override=True)
REDIS_URL = os.getenv("REDIS_URL")

# 지난번 확인 결과: redis:// 가 빠져있었다면 여기서 보정해줍니다.
if REDIS_URL and not REDIS_URL.startswith("redis://"):
    REDIS_URL = f"redis://{REDIS_URL}"

# 수정된 초기화 코드
semantic_cache = RedisSemanticCache(
    redis_url=REDIS_URL,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small")
    # distance_threshold는 과감히 삭제하세요! 
)

print("성공적으로 초기화되었습니다!")

성공적으로 초기화되었습니다!


In [16]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_core.globals import set_llm_cache
from langchain_redis import RedisSemanticCache          # 변경
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
REDIS_URL = os.environ['REDIS_URL']

semantic_cache = RedisSemanticCache(
    redis_url=REDIS_URL,
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),  # 파라미터명 변경
    distance_threshold=0.2                                         # 파라미터명 변경
)

set_llm_cache(semantic_cache)
llm = ChatOpenAI(model="gpt-4o-mini")
print("RedisSemanticCache 연결 완료")

RedisSemanticCache 연결 완료


In [17]:
%%time
response = llm.invoke([HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")])
print(response.content)

코스피 지수는 한국 거래소에서 상장된 주식들의 주가를 기초로 계산된 주식 시장 지표로, 한국 경제의 전반적인 건강 상태를 나타냅니다.
CPU times: total: 15.6 ms
Wall time: 2.68 s


In [18]:
%%time
response = llm.invoke([HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")])
print(response.content)

코스피 지수는 한국 거래소에서 상장된 주식들의 주가를 기초로 계산된 주식 시장 지표로, 한국 경제의 전반적인 건강 상태를 나타냅니다.
CPU times: total: 0 ns
Wall time: 252 ms


In [19]:
%%time
response = llm.invoke([HumanMessage(content="코스피가 뭔지 간단히 설명해줘.")])
print(response.content)

코스피 지수는 한국 거래소에서 상장된 주식들의 주가를 기초로 계산된 주식 시장 지표로, 한국 경제의 전반적인 건강 상태를 나타냅니다.
CPU times: total: 15.6 ms
Wall time: 386 ms


In [21]:
semantic_cache.clear()

In [22]:
import time
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import langchain

# 캐시 활성화 확인 (이게 있어야 작동합니다!)
langchain.llm_cache = semantic_cache 

llm = ChatOpenAI(model="gpt-4o")

def ask_question(question):
    start_time = time.time()
    
    # 질문 던지기
    response = llm.invoke([HumanMessage(content=question)])
    
    end_time = time.time()
    print(f"질문: {question}")
    print(f"답변: {response.content[:50]}...") # 앞부분만 출력
    print(f"소요 시간: {end_time - start_time:.2f}초")
    print("-" * 30)

# --- 테스트 실행 ---

# 1. 첫 질문 (캐시 저장됨)
print("[1차 실행 - 캐시 저장 중]")
ask_question("양자역학이 뭐야? 초보자용으로 짧게 설명해줘.")

# 2. 다른 표현의 질문 (캐시 적중 기대)
print("[2차 실행 - 시만틱 캐시 확인]")
ask_question("퀀텀 메카닉스에 대해서 아주 쉽게 요약해줄래?")

# 3. 아예 다른 질문 (캐시 미스 기대)
print("[3차 실행 - 새로운 질문]")
ask_question("가장 맛있는 사과 고르는 법 알려줘.")

[1차 실행 - 캐시 저장 중]
질문: 양자역학이 뭐야? 초보자용으로 짧게 설명해줘.
답변: 양자역학은 아주 작은 입자들, 예를 들어 원자나 전자 같은 미시적인 세계를 설명하는 물리학...
소요 시간: 2.70초
------------------------------
[2차 실행 - 시만틱 캐시 확인]
질문: 퀀텀 메카닉스에 대해서 아주 쉽게 요약해줄래?
답변: 양자역학은 아주 작은 입자들, 예를 들어 원자나 전자 같은 미시적인 세계를 설명하는 물리학...
소요 시간: 0.13초
------------------------------
[3차 실행 - 새로운 질문]
질문: 가장 맛있는 사과 고르는 법 알려줘.
답변: 양자역학은 아주 작은 입자들, 예를 들어 원자나 전자 같은 미시적인 세계를 설명하는 물리학...
소요 시간: 0.14초
------------------------------
